In [1]:
import pandas as pd
import numpy as np
import glob
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [2]:
# Loading .pkl files

DATA_PATH = "data/"   # folder containing daily .pkl files

files = sorted(glob.glob(os.path.join(DATA_PATH, "*.pkl")))

if len(files) == 0:
    raise FileNotFoundError("No .pkl files found inside data/ folder")

df_list = []

for file in files:
    df_day = pd.read_pickle(file)
    df_list.append(df_day)

# Combine all days
df = pd.concat(df_list, ignore_index=True)
df = df.reset_index(drop=True)


# Ensure datetime column
if "TX_DATETIME" not in df.columns:
    df = df.reset_index()

df["TX_DATETIME"] = pd.to_datetime(df["TX_DATETIME"])
df = df.sort_values("TX_DATETIME")

print("Total Transactions:", len(df))
print("Fraud Distribution:")
print(df["TX_FRAUD"].value_counts())

Total Transactions: 1754155
Fraud Distribution:
TX_FRAUD
0    1739474
1      14681
Name: count, dtype: int64


In [3]:
df.head()

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


In [4]:
# Time-based features

df["tx_hour"] = df["TX_DATETIME"].dt.hour
df["tx_day"] = df["TX_DATETIME"].dt.day
df["tx_dayofweek"] = df["TX_DATETIME"].dt.dayofweek
df["is_weekend"] = df["tx_dayofweek"].isin([5, 6]).astype(int)

df["TX_TIME_DAYS"] = (
    df["TX_DATETIME"] - df["TX_DATETIME"].min()
).dt.days

df["TX_TIME_SECONDS"] = (
    df["TX_DATETIME"].dt.hour * 3600 +
    df["TX_DATETIME"].dt.minute * 60 +
    df["TX_DATETIME"].dt.second
)


In [5]:
# HIGH AMOUNT RULE
df["high_amount_flag"] = (df["TX_AMOUNT"] > 220).astype(int)

In [6]:
# Sort properly
df = df.sort_values(["TERMINAL_ID", "TX_DATETIME"])

# Set datetime index temporarily
original_datetime = df["TX_DATETIME"]
df = df.set_index("TX_DATETIME")

# 28-day rolling fraud count
df["term_fraud_count_28d"] = (
    df.groupby("TERMINAL_ID")["TX_FRAUD"]
      .rolling("28D")
      .sum()
      .reset_index(level=0, drop=True)
)

# 28-day transaction count
df["term_tx_count_28d"] = (
    df.groupby("TERMINAL_ID")["TX_AMOUNT"]
      .rolling("28D")
      .count()
      .reset_index(level=0, drop=True)
)

# Reset index back
df = df.reset_index()

df["term_fraud_rate_28d"] = (
    df["term_fraud_count_28d"] /
    (df["term_tx_count_28d"] + 1)
)


In [7]:
df = df.sort_values(["CUSTOMER_ID", "TX_DATETIME"])
df = df.set_index("TX_DATETIME")

df["cust_avg_amt_14d"] = (
    df.groupby("CUSTOMER_ID")["TX_AMOUNT"]
      .rolling("14D")
      .mean()
      .reset_index(level=0, drop=True)
)

df["cust_tx_count_14d"] = (
    df.groupby("CUSTOMER_ID")["TX_AMOUNT"]
      .rolling("14D")
      .count()
      .reset_index(level=0, drop=True)
)

df = df.reset_index()

df["cust_amt_ratio"] = (
    df["TX_AMOUNT"] /
    (df["cust_avg_amt_14d"] + 1)
)



In [8]:
 # ADDITIONAL FEATURES

df["log_tx_amount"] = np.log1p(df["TX_AMOUNT"])

df = df.fillna(0)

C:\Users\kanch\AppData\Local\Temp\ipykernel_10636\1765294120.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


In [9]:
# FINAL FEATURE LIST (NO LEAKAGE)

FINAL_FEATURES = [
    # --- Amount behaviour
    "TX_AMOUNT",
    "high_amount_flag",
    "log_tx_amount",

    # --- Terminal behaviour (28 days)
    "term_fraud_rate_28d",
    "term_fraud_count_28d",

    # --- Customer behaviour (14 days)
    "cust_avg_amt_14d",
    "cust_tx_count_14d",
    "cust_amt_ratio",

    # --- Time context
    "TX_TIME_DAYS",
    "tx_dayofweek",
    "is_weekend"
]


X = df[FINAL_FEATURES]
y = df["TX_FRAUD"]




In [10]:
# TIME-AWARE SPLIT (80% train, 20% test)

split_date = df["TX_DATETIME"].quantile(0.8)

X_train = X[df["TX_DATETIME"] <= split_date]
y_train = y[df["TX_DATETIME"] <= split_date]

X_test  = X[df["TX_DATETIME"] > split_date]
y_test  = y[df["TX_DATETIME"] > split_date]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1403324
Test size: 350831


In [11]:
# Base model pipeline
base_model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"))
])
base_model_pipeline.fit(X_train, y_train)

y_pred = base_model_pipeline.predict(X_test)
y_proba = base_model_pipeline.predict_proba(X_test)[:, 1]
print("\nBaseline Model Performance")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))



Baseline Model Performance
              precision    recall  f1-score   support

           0       0.99      1.00      1.00    347681
           1       0.93      0.30      0.46      3150

    accuracy                           0.99    350831
   macro avg       0.96      0.65      0.73    350831
weighted avg       0.99      0.99      0.99    350831

ROC-AUC: 0.9755338388779389


In [15]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

y_pred = base_model_pipeline.predict(X_test)
y_prob = base_model_pipeline.predict_proba(X_test)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Confusion Matrix:
 [[347609     72]
 [  2194    956]]
Precision: 0.9299610894941635
Recall: 0.3034920634920635
F1 Score: 0.4576352321685017
ROC-AUC: 0.9755338388779389


In [16]:
threshold = 0.3   # try 0.25 – 0.4

y_pred_custom = (y_prob >= threshold).astype(int)

print("Recall (custom threshold):", recall_score(y_test, y_pred_custom))
print("Precision (custom threshold):", precision_score(y_test, y_pred_custom))


Recall (custom threshold): 0.6247619047619047
Precision (custom threshold): 0.6527363184079602


In [19]:
import pandas as pd

importances = base_model_pipeline.named_steps["rf"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df)


                 Feature  Importance
3    term_fraud_rate_28d    0.495480
4   term_fraud_count_28d    0.329040
7         cust_amt_ratio    0.061863
2          log_tx_amount    0.032463
0              TX_AMOUNT    0.028058
5       cust_avg_amt_14d    0.019622
8           TX_TIME_DAYS    0.011686
6      cust_tx_count_14d    0.010489
1       high_amount_flag    0.006263
9           tx_dayofweek    0.004231
10            is_weekend    0.000805


In [20]:
import joblib
joblib.dump(base_model_pipeline, "fraud_detection_final_model.pkl")


['fraud_detection_final_model.pkl']